# NNCoxKL: Flexible Transfer Learning in Deep Cox Models

## Motivation
Consider a center with a **small internal cohort** that wants to train a deep Cox model for time-to-event prediction.

**The challenge:** the internal dataset is too small for reliable deep learning, which can lead to overfitting, unstable hyperparameter tuning, and weak generalization.

**What we have access to:** external prognostic information from an existing clinical scoring system. But:
- Raw external patient-level data may **not be shareable**
- The external information is **point-based**, not directly a model-ready survival probability/risk function

NNCoxKL is designed for this setting by flexibly transferring external risk information into deep Cox modeling via generalized Kullback-Leibler regularization.

## Our Method: NNCoxKL
To capture information across failure times $t_1, \ldots, t_K$, we define the accumulated generalized KL distance as

$$
\mathrm{D}(\tilde{\mathbf{r}} \parallel \mathbf{r})=\sum_{k=1}^{K} \mathrm{d}(\mathbf{p}_k \Vert \mathbf{q}_k),
$$

which measures the discrepancy between external risk scores $\tilde{\mathbf{r}}=\{\tilde r(\mathbf{Z}_i), i=1,\ldots,n\}$ and internal risk scores $\mathbf{r}=\{r(\mathbf{Z}_i,\boldsymbol\beta), i=1,\ldots,n\}$.

To integrate internal and external information while allowing potential mismatch, we maximize a penalized log-partial likelihood:

$$
\ell_{\eta}(\boldsymbol\beta)=\ell(\boldsymbol\beta)-\eta\,\mathrm{D}(\tilde{\mathbf{r}} \parallel \mathbf{r}),
$$

where $\eta\ge 0$ controls how strongly external information is borrowed.

- $\eta=0$: recovers the internal-only Cox objective
- Larger $\eta$: enforces stronger alignment with external risk structure
- $\eta$ can be selected via Optuna or cross-validation

## Quick Start

Below is a simulation code:
It demonstrates the core NNCoxKL pipeline:
1) train an external neural Cox model,
2) extract external risk scores,
3) train an internal NNCoxKL model with KL regularization,
4) evaluate performance on test data.

In [3]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from NNCoxKL.model import MLPDynamic
from NNCoxKL.train_utils import Trainer
from NNCoxKL.dataset import Data
from NNCoxKL.evaluate_utils import Evaluator


def model_device(model):
    return next(model.parameters()).device


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def simulate_survival_data(N, p_bin_prob=0.5, noise_level=1.5, seed=42, external=False):
    """Simplified simulation adapted from setting2 script."""
    rng = np.random.default_rng(seed)

    # 10 binary covariates
    Z = rng.binomial(1, p_bin_prob, size=(N, 10)).astype("float32")
    z1, z2, z3, z4, z5 = Z[:, 0], Z[:, 1], Z[:, 2], Z[:, 3], Z[:, 4]
    z6, z7, z8, z9, z10 = Z[:, 5], Z[:, 6], Z[:, 7], Z[:, 8], Z[:, 9]

    beta = np.array([-1.5, -1.0, -1.8, -0.9, 0.6, -0.2], dtype="float32")
    if external:
        beta = np.zeros_like(beta)
    else:
        beta = beta * 0.2

    # Nonlinear log-hazard structure
    xbeta = (
        np.sin(0.5 * np.pi * z1)
        + np.exp(-z2)
        + beta[0] * (z1**2 + z2 * z3)
        + beta[1] * (z2**2)
        + beta[2] * (z3**2 + z1 * z2)
        + beta[3] * (z4**2)
        + beta[4] * (z1 * z3)
        + beta[5] * (z2 * z4 + z2 * z5)
        - 0.3 * z6 + 0.3 * z7 - 0.2 * z8 + 0.5 * z9 + 0.3 * z10
    )

    eta = xbeta + rng.normal(0.0, noise_level, size=N)

    # Event and censoring times
    U = rng.uniform(0, 1, N)
    T_true = -np.log(U) / np.exp(eta)
    C = rng.uniform(0, 0.8, N)

    status = (T_true <= C).astype("float32")
    time = np.where(status == 1, T_true, C).astype("float32")

    df = pd.DataFrame(Z, columns=[f"Z{i+1}" for i in range(10)])
    df["status"] = status
    df["time"] = time
    return df.sort_values("time").reset_index(drop=True)

## 1.1 Data-Generating Process

Both cohorts use the same 10 binary features, but with different feature prevalence to create a distribution shift:

- **Internal cohort**: $Z_j \sim \mathrm{Bernoulli}(0.5)$
- **External cohort**: $Z_j \sim \mathrm{Bernoulli}(0.9)$

This shift is the key heterogeneity in the tutorial example, and motivates transfer learning from external to internal data.

| Dataset | Role | Size |
|---|---|---|
| `data_external` | External cohort | **n = 500** |
| `data_internal` | Internal cohort | **n = 200** |
| `data_test` | Independent evaluation cohort | **n = 1000** |

### Chunk 1: Simulate Data and Preview Examples
Create internal, external, and test cohorts, then use `head()` to inspect example rows.

In [4]:
set_seed(42)

data_internal = simulate_survival_data(N=200, p_bin_prob=0.5, seed=42, external=False)
data_external = simulate_survival_data(N=500, p_bin_prob=0.9, seed=43, external=True)
data_test = simulate_survival_data(N=1000, p_bin_prob=0.5, seed=44, external=False)

print("Internal cohort (head):")
print(data_internal.head())
print("\nExternal cohort (head):")
print(data_external.head())
print("\nTest cohort (head):")
print(data_test.head())

Internal cohort (head):
    Z1   Z2   Z3   Z4   Z5   Z6   Z7   Z8   Z9  Z10  status      time
0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  1.0  1.0  1.0     1.0  0.000102
1  1.0  0.0  1.0  0.0  1.0  1.0  0.0  0.0  1.0  1.0     1.0  0.000225
2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0     1.0  0.000970
3  0.0  0.0  0.0  0.0  0.0  1.0  1.0  0.0  1.0  1.0     1.0  0.001009
4  1.0  1.0  0.0  1.0  1.0  0.0  1.0  0.0  1.0  0.0     1.0  0.001165

External cohort (head):
    Z1   Z2   Z3   Z4   Z5   Z6   Z7   Z8   Z9  Z10  status      time
0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.0  1.0     0.0  0.000093
1  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  0.000141
2  1.0  0.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  0.000314
3  0.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  0.000318
4  1.0  0.0  1.0  0.0  1.0  1.0  0.0  1.0  1.0  0.0     1.0  0.000359

Test cohort (head):
    Z1   Z2   Z3   Z4   Z5   Z6   Z7   Z8   Z9  Z10  status      time
0  1.0  1.0  0.0  0.

### Chunk 2: Train External  Model
Train a neural Cox model on the external cohort. We use `eta=0` so this is standard Cox-style training.

In [5]:
X_ext = data_external.drop(columns=["time", "status"]).values.astype("float32")
y_ext = data_external[["status", "time"]].values.astype("float32")

x_ext_tr, x_ext_va, y_ext_tr, y_ext_va = train_test_split(
    X_ext, y_ext, test_size=0.2, random_state=42
)

rs_dummy_tr = np.zeros(x_ext_tr.shape[0], dtype="float32")
rs_dummy_va = np.zeros(x_ext_va.shape[0], dtype="float32")

external_model = MLPDynamic(input_dim=x_ext_tr.shape[1], hidden_dim=64, output_dim=1, num_layers=2, num_dr=0.1)
trainer_external = Trainer(external_model, optimizer=torch.optim.AdamW(external_model.parameters(), weight_decay=1e-4))

_ = trainer_external.fit(
    Data(x_ext_tr, y_ext_tr, rs_dummy_tr),
    Data(x_ext_va, y_ext_va, rs_dummy_va),
    eta_list=[0],
    alpha=1.0,
    batch_size=64,
    num_epoch=120,
    learning_rate=1e-3,
    patience=20,
)

external_model = external_model.to("cpu")
external_model.eval()

use pytorch-cuda for training on cuda
use pytorch-cuda for training.
loss for epoch 0 with eta 0: 2.702, val_loss: 2.822
loss for epoch 1 with eta 0: 2.492, val_loss: 2.783
loss for epoch 2 with eta 0: 2.437, val_loss: 2.751
loss for epoch 3 with eta 0: 2.428, val_loss: 2.741
loss for epoch 4 with eta 0: 2.400, val_loss: 2.740
loss for epoch 5 with eta 0: 2.369, val_loss: 2.744
loss for epoch 6 with eta 0: 2.391, val_loss: 2.753
loss for epoch 7 with eta 0: 2.412, val_loss: 2.760
loss for epoch 8 with eta 0: 2.400, val_loss: 2.766
loss for epoch 9 with eta 0: 2.387, val_loss: 2.774
loss for epoch 10 with eta 0: 2.388, val_loss: 2.774
loss for epoch 11 with eta 0: 2.373, val_loss: 2.774
loss for epoch 12 with eta 0: 2.377, val_loss: 2.771
loss for epoch 13 with eta 0: 2.356, val_loss: 2.771
loss for epoch 14 with eta 0: 2.381, val_loss: 2.765
loss for epoch 15 with eta 0: 2.380, val_loss: 2.758
loss for epoch 16 with eta 0: 2.342, val_loss: 2.759
loss for epoch 17 with eta 0: 2.356, val

MLPDynamic(
  (layers): ModuleList(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=64, bias=True)
  )
  (output_layer): Linear(in_features=64, out_features=1, bias=False)
  (batchnorms): ModuleList(
    (0-1): 2 x BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (dropouts): ModuleList(
    (0-1): 2 x Dropout(p=0.1, inplace=False)
  )
)

### Chunk 3: Build Internal Split and External Risk Scores
Use the trained external model to generate external risk scores for internal train/validation sets.

In [6]:
X_int = data_internal.drop(columns=["time", "status"]).values.astype("float32")
y_int = data_internal[["status", "time"]].values.astype("float32")

x_tr, x_va, y_tr, y_va = train_test_split(X_int, y_int, test_size=0.2, random_state=42)

with torch.no_grad():
    x_tr_t = torch.tensor(x_tr, dtype=torch.float32, device=model_device(external_model))
    x_va_t = torch.tensor(x_va, dtype=torch.float32, device=model_device(external_model))
    rs_tr = external_model(x_tr_t).detach().cpu().numpy().flatten()
    rs_va = external_model(x_va_t).detach().cpu().numpy().flatten()

### Chunk 4: Train Internal NNCoxKL
Train the internal model using internal outcomes plus KL regularization against transferred external risk scores.

In [7]:
eta = 0.5  # Increase/decrease to control external borrowing strength

internal_model = MLPDynamic(input_dim=x_tr.shape[1], hidden_dim=64, output_dim=1, num_layers=2, num_dr=0.1)
trainer_internal = Trainer(internal_model, optimizer=torch.optim.AdamW(internal_model.parameters(), weight_decay=1e-4))

_ = trainer_internal.fit(
    Data(x_tr, y_tr, rs_tr),
    Data(x_va, y_va, rs_va),
    eta_list=[eta],
    eta=eta,
    alpha=1.0,
    batch_size=64,
    num_epoch=150,
    learning_rate=5e-4,
    patience=20,
)

use pytorch-cuda for training on cuda
use pytorch-cuda for training.
loss for epoch 0 with eta 0.5: 1.963, val_loss: 1.559
loss for epoch 1 with eta 0.5: 2.165, val_loss: 1.540
loss for epoch 2 with eta 0.5: 2.122, val_loss: 1.525
loss for epoch 3 with eta 0.5: 1.914, val_loss: 1.510
loss for epoch 4 with eta 0.5: 1.935, val_loss: 1.497
loss for epoch 5 with eta 0.5: 1.902, val_loss: 1.486
loss for epoch 6 with eta 0.5: 1.857, val_loss: 1.475
loss for epoch 7 with eta 0.5: 1.864, val_loss: 1.466
loss for epoch 8 with eta 0.5: 1.807, val_loss: 1.457
loss for epoch 9 with eta 0.5: 1.819, val_loss: 1.449
loss for epoch 10 with eta 0.5: 1.871, val_loss: 1.443
loss for epoch 11 with eta 0.5: 1.864, val_loss: 1.438
loss for epoch 12 with eta 0.5: 1.711, val_loss: 1.435
loss for epoch 13 with eta 0.5: 1.685, val_loss: 1.432
loss for epoch 14 with eta 0.5: 1.834, val_loss: 1.430
loss for epoch 15 with eta 0.5: 1.717, val_loss: 1.428
loss for epoch 16 with eta 0.5: 1.667, val_loss: 1.428
loss f

### Chunk 5: Evaluate on Test Data
Move model to CPU for safe evaluation and report C-index/loss.

In [8]:
internal_model = internal_model.to("cpu")
internal_model.eval()

X_test = data_test.drop(columns=["time", "status"]).values.astype("float32")
y_test = data_test[["status", "time"]].values.astype("float32")

evaluator = Evaluator()
c_index_test, loss_test = evaluator.eval_single(internal_model, Data(X_test, y_test))

print(f"NNCoxKL test C-index: {float(c_index_test):.4f}")
print(f"NNCoxKL test loss:    {float(loss_test):.4f}")

NNCoxKL test C-index: 0.6375
NNCoxKL test loss:    3.3094
